# 환경 세팅

.env 파일을 로드하여 환경변수에 등록합니다. 노트북을 위한 실행 경로를 세팅합니다.

In [ ]:
import os, sys
from dotenv import load_dotenv

load_dotenv(override=True)
#load_dotenv("/mnt/d/agent_research/github-repos/home/AAWS_project/.env", override=True)

# PROJECT_ROOT를 .env에서 읽기 (없으면 현재 디렉토리)
project_root = os.getenv("PROJECT_ROOT", os.getcwd())

# 프로젝트 루트가 유효하지 않으면, 현재 위치에서 상위로 찾기
if not os.path.exists(os.path.join(project_root, "app")):
    # 상위 디렉토리 탐색
    current = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(current, "app")):
            project_root = current
            break
        current = os.path.dirname(current)

# Working Directory 설정
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import app

print(f"✅ Project Root: {project_root}")
print(f"✅ Working Directory: {os.getcwd()}")

### 웹 데이터 수집 방식 비교 (HTTP vs 브라우저 자동화)

웹에서 데이터를 수집하거나 상호작용하는 방법은 크게 두 가지로 나뉩니다.

### 방법 A: HTTP 요청 기반 수집 (API, Requests, BeautifulSoup)
* **작동 방식:** 특정 URL에 패킷 요청을 직접 보내 서버가 반환하는 HTML 텍스트 파일만 가볍게 받아옵니다.
* **장점:** 매우 빠르고 메모리/CPU 리소스를 거의 사용하지 않습니다.
* **단점/한계:**
  * **동적 렌더링 불가:** 최근의 현대적인 웹사이트(React, Vue, Next.js 등)는 초기 HTML이 빈 껍데기며, JavaScript가 실행되어야 비로소 화면이 그려집니다. HTTP 요청만으로는 빈 화면만 수집하게 됩니다.
  * **인터랙션 불가:** 로그인 상태 유지, 마우스 스크롤을 내려 데이터 추가 로드(Infinite Scroll), 팝업 창 클릭 등이 불가능합니다.
  * **보안 차단:** 봇 방지 솔루션(CAPTCHA, Cloudflare)에 의해 즉시 차단되기 쉽습니다.

### 방법 B: 실제 브라우저 자동화 (Playwright, Selenium)
* **작동 방식:** 메모리 상에 실제 웹 브라우저 프로세스를 실행하고, 프로그램 코드로 브라우저를 직접 조종합니다.
* **장점:** 
  * 화면 안의 모든 JavaScript가 실행되므로 **사람이 눈으로 보는 것과 100% 동일한 동적 웹 화면**을 확보합니다.
  * 클릭, 키보드 입력, 스크롤, 멀티 탭 이동 등 사람이 할 수 있는 모든 인터랙션을 구현할 수 있습니다.
  * 쿠키와 세션을 활용해 복잡한 로그인 세션을 쉽게 우회합니다.
* **단점:** 브라우저 엔진을 직접 띄우기 때문에 리소스(메모리, CPU) 소모가 큽니다.
> 💡 **결론:** AI 에이전트가 웹 공간에서 인간처럼 문제를 해결하려면, 단순 텍스트 패킷이 아니라 **실제 웹 브라우저를 다룰 수 있는 제어권**이 반드시 필요합니다.



| 비교 항목 | HTTP 요청 (requests, BeautifulSoup) | 브라우저 자동화 (Playwright, Selenium) |
| :--- | :--- | :--- |
| **작동 방식** | 서버에 직접 HTTP 요청을 보내 원시 HTML 텍스트를 수집 | 실제 브라우저 엔진을 코드로 기동하여 물리적으로 조작 |
| **성능 및 자원** | ⚡ 매우 빠르고 가벼움 (최소한의 리소스 소모) | 🐢 상대적으로 무겁고 브라우저 구동 자원 소모 |
| **JavaScript 렌더링**| ❌ 불가능 (React/Vue 등 동적 웹은 빈 HTML만 받아옴) | ⭕ 동적 CSR / AJAX 데이터를 완벽히 실행하여 수집 가능 |
| **사용자 인터랙션**| ❌ 클릭, 키보드 입력, 스크롤, 로그인 등 제어 불가 | ⭕ 인간과 동일한 클릭, 스크롤, 팝업 제어 등 가능 |
| **세션 및 상태** | ❌ 세션 및 로그인 유지가 매우 복잡하고 까다로움 | ⭕ 브라우저 세션(쿠키, 세션 상태)이 자연스럽게 유지됨 |

---

### 기반 인프라 및 핵심 개념 요약

| 구분 | 인프라 / 기술 | 핵심 정의 및 역할 |
| :---: | :--- | :--- |
| **엔진** | **Chromium** | Chrome, Edge 등의 근간이 되는 브라우저 엔진. 에이전트의 **'실제 손과 발'**이 되는 브라우저입니다. |
| **도구** | **Playwright** | MS가 개발한 브라우저 제어 라이브러리. 압도적인 속도와 요소 등장 대기(**Auto-wait**) 기능을 지원합니다. |
| **옵션** | **Headless 모드** | 화면(GUI) 없이 백그라운드 프로세스로만 구동하는 모드. 클라우드 서버나 Docker 배포 시 자원을 절약합니다. |
| **옵션** | **Headful (GUI) 모드** | 눈에 보이는 일반 크롬 창 형태로 브라우저를 띄우는 모드. 에이전트 동작 모니터링 및 **디버깅 시 필수**입니다. |
| **가상화**| **가상 디스플레이 (Xvfb)**| 메모리 상에 가짜 모니터를 생성하는 기술. WSL2, Docker 등 **디스플레이 장치가 없는 환경의 실행 크래시를 방지**합니다. |
| **송출** | **noVNC** | 가상 화면을 웹 브라우저 창으로 송출하는 기술. 가상 리눅스 환경에서도 **에이전트 조작 화면을 라이브 시청**하게 돕습니다. |

---

In [ ]:
import asyncio
from browser_use import Agent, Browser, ChatGoogle

import nest_asyncio

# 주피터 노트북 환경에서 비동기 루프 중복 실행 오류를 방지하기 위해 사용합니다.
nest_asyncio.apply()

# 가상 디스플레이(noVNC) 번호를 지정합니다.
# 이 설정이 있어야 GUI 브라우저가 가상 화면(:1)에 나타납니다. .env에서 설정하지 않았다면 아래 주석을 해제하고 실행하세요.
# os.environ["DISPLAY"] = ":1"

In [ ]:
import asyncio
from playwright.async_api import async_playwright
import os

async def main():
    # DISPLAY 환경 변수가 설정되어 있는지 확인 (기본값 :1)
    #os.environ["DISPLAY"] = os.environ.get("DISPLAY", ":1")
    
    async with async_playwright() as p:
        # 1. 브라우저 실행 시 '창 최대화' 인자 추가
        # --window-size 설정을 통해 초기 창 크기를 가상 디스플레이와 맞춥니다.
        browser = await p.chromium.launch(
            headless=False,
            args=["--start-maximized", "--window-size=1280,750"]
        )
        
        # 2. Viewport 크기를 'None'으로 설정하여 브라우저 창 크기에 맞게 조절하거나
        # 가상 디스플레이 크기(1280x800)로 직접 고정합니다.
        context = await browser.new_context(viewport={"width": 1280, "height": 750})
        page = await context.new_page()
        
        print("브라우저를 열고 구글로 이동합니다...")
        await page.goto("https://www.google.com")
        
        # 화면을 확인할 수 있게 잠시 대기
        await asyncio.sleep(10) 
        await browser.close()

if __name__ == "__main__":
    asyncio.run(main())